# UNSW-NB15 Intrusion Detection Model

A simple binary IDS: classifies each network flow as **normal (0)** or **attack (1)** using a Random Forest.

**Data:** download `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv` from the
[UNSW-NB15 page](https://research.unsw.edu.au/projects/unsw-nb15-dataset) (CSV Files → Training and Testing Sets)
and put them in a `data/` folder next to this notebook.

In [1]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

## 1. Load the data

In [2]:
train = pd.read_csv("data/UNSW_NB15_training-set.csv")
test = pd.read_csv("data/UNSW_NB15_testing-set.csv")

print("Train:", train.shape, " Test:", test.shape)
train.head()

Train: (175341, 45)  Test: (82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [3]:
# Class balance (0 = normal, 1 = attack) and attack types in the training set
print(train["label"].value_counts(), "\n")
print(train["attack_cat"].value_counts())

label
1    119341
0     56000
Name: count, dtype: int64 

attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


## 2. Prepare features

- Drop `id` (row number) and `attack_cat` (attack type — it would leak the answer for binary classification).
- `proto`, `service` and `state` are text columns, so they are one-hot encoded; the other features are used as-is.

In [4]:
drop = ["id", "attack_cat", "label"]
X_train, y_train = train.drop(columns=drop), train["label"]
X_test, y_test = test.drop(columns=drop), test["label"]

cat_cols = ["proto", "service", "state"]
pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough",
)

## 3. Train the model

In [5]:
model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)),
])

model.fit(X_train, y_train)

/Users/vatto/Documents/CMPE_272/NB15_model/.venv/lib/python3.9/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('pre',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))])

## 4. Evaluate on the test set

In [6]:
pred = model.predict(X_test)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["normal", "attack"]))

[[27030  9970]
 [  640 44692]]
              precision    recall  f1-score   support

      normal       0.98      0.73      0.84     37000
      attack       0.82      0.99      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.86     82332
weighted avg       0.89      0.87      0.87     82332



In [7]:
# How often each kind of traffic is flagged as an attack
# (for Normal this is the false-alarm rate; for attack types it is the detection rate)
results = test.assign(flagged=pred)
by_cat = results.groupby("attack_cat")["flagged"].agg(rows="count", flagged_pct="mean")
by_cat["flagged_pct"] = (by_cat["flagged_pct"] * 100).round(1)
by_cat.sort_values("rows", ascending=False)

,rows,flagged_pct
attack_cat,,
Normal,37000,26.9
Generic,18871,100.0
Exploits,11132,99.6
Fuzzers,6062,90.3
DoS,4089,99.9
Reconnaissance,3496,100.0
Analysis,677,99.7
Backdoor,583,100.0
Shellcode,378,99.5


## 5. Save the model

In [8]:
joblib.dump(model, "model.joblib")

['model.joblib']

## 6. Predict on new data

The input CSV must have the same feature columns as UNSW-NB15. `id`, `attack_cat` and `label` are ignored if present.
To use a saved model without retraining, run the imports cell, then `model = joblib.load("model.joblib")`.

In [9]:
def predict_file(in_path, out_path="predictions.csv"):
    df = pd.read_csv(in_path)
    X = df.drop(columns=["id", "attack_cat", "label"], errors="ignore")
    df["prediction"] = pd.Series(model.predict(X)).map({0: "normal", 1: "attack"})
    df["attack_probability"] = model.predict_proba(X)[:, 1].round(3)
    df.to_csv(out_path, index=False)
    return df

# Example: predict on the first 10 rows of the test set
test.head(10).to_csv("sample.csv", index=False)
predict_file("sample.csv", "sample_predictions.csv")[["proto", "attack_cat", "label", "prediction", "attack_probability"]]

,proto,attack_cat,label,prediction,attack_probability
0,udp,Normal,0,attack,0.769
1,udp,Normal,0,attack,0.870
2,udp,Normal,0,attack,0.748
3,udp,Normal,0,attack,0.850
4,udp,Normal,0,attack,0.730
5,udp,Normal,0,attack,0.748
6,udp,Normal,0,attack,0.657
7,udp,Normal,0,attack,0.740
8,arp,Normal,0,normal,0.000
9,arp,Normal,0,normal,0.000
